## Task 3: Soft Label Loss

**Goal:** Use Label Smoothing BCE as an alternative to cross-entropy on the same imbalanced dataset.

1. Implement `LabelSmoothingBinaryCrossEntropy`:

```python
class LabelSmoothingBinaryCrossEntropy(nn.Module):
    """
    Label Smoothing Binary Cross-Entropy.
    Replaces hard binary targets {0, 1} with smoothed targets:
      y_smoothed = y * (1 - eps) + (1 - y) * eps
    Prevents overconfidence and improves calibration.
    Works as a regulariser for noisy labels.
    Args:
        eps (float): smoothing strength. Typical range: 0.05 – 0.15
        reduction (str): 'mean' | 'sum' | 'none'
    """
    def __init__(self, eps: float = 0.1, reduction: str = "mean"):
        super().__init__()
        self.eps       = eps
        self.reduction = reduction

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        targets          = targets.float()
        smoothed_targets = targets * (1.0 - self.eps) + (1.0 - targets) * self.eps
        return F.binary_cross_entropy_with_logits(logits, smoothed_targets,
                                                   reduction=self.reduction)
```

2. Implement, train, evaluate and compare analogously to Task 2 — for several values of the smoothing coefficient `eps`.

**Assignment:** Implement and train a classifier on a strongly imbalanced dataset — compare classification quality for cross-entropy and Soft Label Loss.

In [5]:
IMBALANCE_RATIO = 0.95
FOCAL_ALPHA = 0.75   # weight for the positive class
FOCAL_GAMMA = 2.0    # focusing on hard examples

BATCH_SIZE   = 128
EPOCHS       = 1000
LR           = 1e-3
WEIGHT_DECAY = 1e-4

In [6]:
from utils import make_imbalanced_dataset, BinaryClassifierMLP, make_loaders
import torch

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
model = BinaryClassifierMLP().to(device)

X, y = make_imbalanced_dataset(imbalance_ratio=IMBALANCE_RATIO)
print(f"Class 0: {(y == 0).sum():.0f}  Class 1: {(y == 1).sum():.0f}")

train_loader, val_loader, test_loader = make_loaders(X, y, batch_size=BATCH_SIZE, squeeze_y=True)

Class 0: 11288  Class 1: 712


In [7]:
from utils import train_baseline
from utils import LabelSmoothingBinaryCrossEntropy
import torch.nn as nn

print("Training with BCE: ")
model_bce = train_baseline(model, train_loader, val_loader,
                           criterion=nn.BCEWithLogitsLoss(), device=device,
                           weight_decay=WEIGHT_DECAY, epochs=EPOCHS, lr=LR
                           )

print("\nTraining with Label Smoothing BCE: ")
model_focal = train_baseline(model, train_loader, val_loader,
                             criterion=LabelSmoothingBinaryCrossEntropy(),
                             device=device, weight_decay=WEIGHT_DECAY, epochs=EPOCHS, lr=LR
                             )

Training with BCE: 
Epoch 100/1000  train=0.0400  val=0.1434
Epoch 200/1000  train=0.0169  val=0.1929
Epoch 300/1000  train=0.0164  val=0.2378
Epoch 400/1000  train=0.0122  val=0.2626
Epoch 500/1000  train=0.0152  val=0.2675
Epoch 600/1000  train=0.0069  val=0.2743
Epoch 700/1000  train=0.0083  val=0.2935
Epoch 800/1000  train=0.0076  val=0.3069
Epoch 900/1000  train=0.0077  val=0.3274
Epoch 1000/1000  train=0.0117  val=0.3330

Training with Label Smoothing BCE: 
Epoch 100/1000  train=0.3509  val=0.3646
Epoch 200/1000  train=0.3460  val=0.3660
Epoch 300/1000  train=0.3428  val=0.3670
Epoch 400/1000  train=0.3409  val=0.3680
Epoch 500/1000  train=0.3399  val=0.3675
Epoch 600/1000  train=0.3395  val=0.3685
Epoch 700/1000  train=0.3392  val=0.3679
Epoch 800/1000  train=0.3362  val=0.3698
Epoch 900/1000  train=0.3364  val=0.3702
Epoch 1000/1000  train=0.3360  val=0.3693


In [8]:
from utils import get_probs, compute_clf_metrics

y_true, probs_bce = get_probs(model_bce, test_loader, device=device)
y_true, probs_focal = get_probs(model_focal, test_loader, device=device)

m_bce = compute_clf_metrics(y_true, probs_bce)
m_focal = compute_clf_metrics(y_true, probs_focal)

print(f"\n{'Metric':<12} {'BCE':>10} {'Focal Loss':>12}")
print("=" * 36)
for key in ["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]:
    print(f"{key:<12} {m_bce[key]:>10.4f} {m_focal[key]:>12.4f}")
print("=" * 36)
print(f"\nBCE        — TP={m_bce['tp']}  FP={m_bce['fp']}  TN={m_bce['tn']}  FN={m_bce['fn']}")
print(f"Focal Loss — TP={m_focal['tp']}  FP={m_focal['fp']}  TN={m_focal['tn']}  FN={m_focal['fn']}")


Metric              BCE   Focal Loss
accuracy         0.9762       0.9762
precision        0.9535       0.9535
recall           0.6074       0.6074
f1               0.7421       0.7421
roc_auc          0.9033       0.9033
pr_auc           0.7843       0.7843

BCE        — TP=82  FP=4  TN=2261  FN=53
Focal Loss — TP=82  FP=4  TN=2261  FN=53
